In [ ]:
# ============================================================================
# OPTIMIZED TELECOM CHURN PREDICTION PIPELINE
# ============================================================================
import pandas as pd
import numpy as np
import re
import warnings
from datetime import datetime
from pathlib import Path
import pickle
from typing import Dict, List, Tuple, Any, Optional

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score, 
                           f1_score, confusion_matrix, precision_recall_curve)
from sklearn.feature_selection import SelectFromModel, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# Gradient boosting imports
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Imbalanced learning
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek

warnings.filterwarnings('ignore')
np.random.seed(42)

# ============================================================================
# CONFIGURATION CLASS
# ============================================================================
class Config:
    """Configuration settings for the pipeline"""
    
    # Data paths
    TRAIN_PATH = './train.csv'
    TEST_PATH = './testA.csv'
    OUTPUT_DIR = Path('./models')
    SUBMISSION_PATH = './submit.csv'
    
    # Model parameters
    N_FOLDS = 5
    TEST_SIZE = 0.2
    RANDOM_STATE = 42
    
    # Feature engineering
    MAX_FEATURES = 200
    FEATURE_SELECTION_THRESHOLD = 0.001
    
    # Model hyperparameters (optimized)
    LIGHTGBM_PARAMS = {
        'n_estimators': 1200,
        'max_depth': 10,
        'learning_rate': 0.025,
        'num_leaves': 50,
        'subsample': 0.85,
        'colsample_bytree': 0.75,
        'min_child_samples': 25,
        'reg_alpha': 0.2,
        'reg_lambda': 0.2,
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbose': -1,
        'force_col_wise': True,
        'boosting_type': 'gbdt',
        'metric': 'auc'
    }
    
    XGBOOST_PARAMS = {
        'n_estimators': 1200,
        'max_depth': 8,
        'learning_rate': 0.025,
        'subsample': 0.85,
        'colsample_bytree': 0.75,
        'min_child_weight': 3,
        'gamma': 0.1,
        'reg_alpha': 0.2,
        'reg_lambda': 0.2,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'eval_metric': 'auc',
        'tree_method': 'hist'
    }
    
    CATBOOST_PARAMS = {
        'iterations': 1200,
        'depth': 9,
        'learning_rate': 0.025,
        'l2_leaf_reg': 2,
        'subsample': 0.85,
        'colsample_bylevel': 0.75,
        'min_data_in_leaf': 20,
        'auto_class_weights': 'Balanced',
        'random_state': RANDOM_STATE,
        'thread_count': -1,
        'verbose': False,
        'eval_metric': 'AUC'
    }

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
class Utils:
    """Utility functions for data processing and model evaluation"""
    
    @staticmethod
    def clean_feature_names(columns: List[str]) -> List[str]:
        """Clean feature names for model compatibility"""
        cleaned = []
        for col in columns:
            cleaned_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
            cleaned_name = re.sub(r'_+', '_', cleaned_name)
            cleaned_name = cleaned_name.strip('_')
            cleaned.append(cleaned_name)
        return cleaned
    
    @staticmethod
    def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
        """Reduce memory usage by optimizing data types"""
        start_mem = df.memory_usage(deep=True).sum() / 1024**2
        
        for col in df.columns:
            col_type = df[col].dtype
            
            if col_type != object:
                c_min = df[col].min()
                c_max = df[col].max()
                
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                        
                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float32)
        
        end_mem = df.memory_usage(deep=True).sum() / 1024**2
        print(f'Memory usage decreased from {start_mem:.2f} MB to {end_mem:.2f} MB '
              f'({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
        
        return df
    
    @staticmethod
    def optimize_threshold(y_true: np.ndarray, y_proba: np.ndarray) -> Tuple[float, float]:
        """Find optimal classification threshold using F1 score"""
        precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        best_f1 = f1_scores[best_idx]
        
        return best_threshold, best_f1

# ============================================================================
# FEATURE ENGINEERING CLASS
# ============================================================================
class FeatureEngineer:
    """Enhanced feature engineering with advanced techniques"""
    
    def __init__(self):
        self.feature_stats = {}
        self.percentile_maps = {}
    
    def create_enhanced_features(self, df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
        """Create comprehensive feature set"""
        data = df.copy()
        print(f"Starting feature engineering... Initial shape: {data.shape}")
        
        # Basic feature engineering
        data = self._create_basic_features(data)
        
        # Advanced ratio features
        data = self._create_ratio_features(data)
        
        # Behavioral pattern features
        data = self._create_behavioral_features(data)
        
        # Statistical features
        data = self._create_statistical_features(data, is_train)
        
        # Interaction features
        data = self._create_interaction_features(data)
        
        # Temporal features (if available)
        data = self._create_temporal_features(data)
        
        print(f"Feature engineering completed. Final shape: {data.shape}")
        return data
    
    def _create_basic_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Create basic engineered features"""
        
        # ARPU efficiency metrics
        if all(col in data.columns for col in ['arpu', 'cm_flux_use']):
            data['arpu_per_mb'] = data['arpu'] / (data['cm_flux_use'] / 1024 + 1)
            data['usage_efficiency'] = np.log1p(data['cm_flux_use']) / (data['arpu'] + 1)
        
        # Tenure-based features
        if 'innet_dura' in data.columns:
            data['tenure_months'] = data['innet_dura'] / 30
            data['is_new_customer'] = (data['innet_dura'] < 90).astype(int)
            data['is_loyal_customer'] = (data['innet_dura'] > 365).astype(int)
        
        # Usage pattern features
        flux_cols = [col for col in data.columns if 'flux' in col.lower()]
        if flux_cols:
            data['total_flux'] = data[flux_cols].sum(axis=1)
            data['flux_variance'] = data[flux_cols].var(axis=1)
            data['flux_skewness'] = data[flux_cols].skew(axis=1)
        
        return data
    
    def _create_ratio_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Create advanced ratio features"""
        
        # Video usage ratios
        video_cols = [col for col in data.columns if 'vid' in col.lower()]
        if video_cols and len(video_cols) > 1:
            data['video_total'] = data[video_cols].sum(axis=1)
            for col in video_cols:
                data[f'{col}_ratio'] = data[col] / (data['video_total'] + 1)
        
        # Bill duration ratios
        bill_cols = [col for col in data.columns if 'bill' in col.lower()]
        if 'cm_tot_bill_dura' in data.columns and 'l3m_avg_bill_dura' in data.columns:
            data['bill_consistency'] = data['cm_tot_bill_dura'] / (data['l3m_avg_bill_dura'] + 1)
        
        return data
    
    def _create_behavioral_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Create behavioral pattern features"""
        
        # Network usage patterns
        if all(col in data.columns for col in ['wday_flux_total', 'nwday_flux_total']):
            total_flux = data['wday_flux_total'] + data['nwday_flux_total']
            data['weekday_usage_ratio'] = data['wday_flux_total'] / (total_flux + 1)
            data['weekend_preference'] = (data['nwday_flux_total'] > data['wday_flux_total']).astype(int)
        
        # Voice vs data preference
        if all(col in data.columns for col in ['cm_local_voice_dura', 'cm_flux_use']):
            data['voice_data_ratio'] = data['cm_local_voice_dura'] / (data['cm_flux_use'] + 1)
            data['is_voice_heavy'] = (data['voice_data_ratio'] > 1).astype(int)
        
        # Engagement score
        engagement_features = ['login_times_m', 'gprs_days', 'open_day_m']
        if all(col in data.columns for col in engagement_features):
            data['engagement_score'] = (
                data['login_times_m'] * 0.4 +
                data['gprs_days'] * 0.3 +
                data['open_day_m'] * 0.3
            )
        
        return data
    
    def _create_statistical_features(self, data: pd.DataFrame, is_train: bool) -> pd.DataFrame:
        """Create statistical features and percentile encodings"""
        
        key_features = ['arpu', 'cm_flux_use', 'innet_dura', 'age']
        
        for feature in key_features:
            if feature in data.columns:
                # Transformations
                data[f'{feature}_log1p'] = np.log1p(data[feature])
                data[f'{feature}_sqrt'] = np.sqrt(data[feature])
                
                # Percentile encoding
                if is_train:
                    try:
                        data[f'{feature}_percentile'] = pd.qcut(
                            data[feature], q=10, labels=False, duplicates='drop'
                        )
                        # Store percentile mapping for test data
                        self.percentile_maps[feature] = pd.qcut(
                            data[feature], q=10, retbins=True, duplicates='drop'
                        )[1]
                    except ValueError:
                        data[f'{feature}_percentile'] = 0
                else:
                    if feature in self.percentile_maps:
                        data[f'{feature}_percentile'] = pd.cut(
                            data[feature], bins=self.percentile_maps[feature], 
                            labels=False, include_lowest=True
                        ).fillna(0)
                    else:
                        data[f'{feature}_percentile'] = 0
        
        return data
    
    def _create_interaction_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Create interaction features between key variables"""
        
        # High-value interactions
        if all(col in data.columns for col in ['arpu', 'cm_flux_use', 'innet_dura']):
            data['arpu_usage_tenure'] = (
                data['arpu'] * np.log1p(data['cm_flux_use']) * np.log1p(data['innet_dura'])
            ) / 10000
        
        # Age-based interactions
        if all(col in data.columns for col in ['age', 'arpu']):
            data['age_arpu_interaction'] = data['age'] * data['arpu'] / 1000
        
        return data
    
    def _create_temporal_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Create temporal pattern features"""
        
        # Day/night usage patterns
        day_cols = [col for col in data.columns if 'day' in col.lower() and 'flux' in col.lower()]
        night_cols = [col for col in data.columns if 'night' in col.lower() and 'flux' in col.lower()]
        
        if day_cols and night_cols:
            data['day_usage'] = data[day_cols].sum(axis=1)
            data['night_usage'] = data[night_cols].sum(axis=1)
            data['day_night_ratio'] = data['day_usage'] / (data['night_usage'] + 1)
            data['is_night_user'] = (data['night_usage'] > data['day_usage']).astype(int)
        
        return data

# ============================================================================
# MODEL TRAINER CLASS
# ============================================================================
class ModelTrainer:
    """Advanced model training with cross-validation and ensemble methods"""
    
    def __init__(self, config: Config):
        self.config = config
        self.models = {}
        self.results = {}
        self.feature_importance = {}
    
    def prepare_data(self, X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, pd.Series]:
        """Prepare data with feature selection and balancing"""
        
        print("Preparing data...")
        
        # Feature selection using multiple methods
        X_selected = self._select_features(X, y)
        
        # Handle class imbalance
        X_balanced, y_balanced = self._handle_imbalance(X_selected, y)
        
        return X_balanced, y_balanced
    
    def _select_features(self, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        """Select most informative features"""
        
        print(f"Starting feature selection from {X.shape[1]} features...")
        
        # Method 1: Mutual information
        mi_scores = mutual_info_classif(X, y, random_state=self.config.RANDOM_STATE)
        mi_threshold = np.percentile(mi_scores, 70)  # Top 30% features
        mi_mask = mi_scores >= mi_threshold
        
        # Method 2: Tree-based selection
        rf_selector = RandomForestClassifier(
            n_estimators=100, random_state=self.config.RANDOM_STATE, n_jobs=-1
        )
        rf_selector.fit(X, y)
        
        tree_selector = SelectFromModel(
            rf_selector, threshold='median', prefit=True
        )
        tree_mask = tree_selector.get_support()
        
        # Combine selections
        combined_mask = mi_mask | tree_mask
        selected_features = X.columns[combined_mask].tolist()
        
        # Limit to max features
        if len(selected_features) > self.config.MAX_FEATURES:
            # Use feature importance to select top features
            importances = rf_selector.feature_importances_[combined_mask]
            importance_df = pd.DataFrame({
                'feature': selected_features,
                'importance': importances
            }).sort_values('importance', ascending=False)
            
            selected_features = importance_df.head(self.config.MAX_FEATURES)['feature'].tolist()
        
        print(f"Selected {len(selected_features)} features")
        return X[selected_features]
    
    def _handle_imbalance(self, X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, pd.Series]:
        """Handle class imbalance using SMOTE + Tomek"""
        
        print(f"Original class distribution: {y.value_counts().to_dict()}")
        
        # Use BorderlineSMOTE with Tomek links for better boundary handling
        sampler = SMOTETomek(
            smote=SMOTE(random_state=self.config.RANDOM_STATE, k_neighbors=5),
            random_state=self.config.RANDOM_STATE
        )
        
        X_resampled, y_resampled = sampler.fit_resample(X, y)
        
        print(f"Resampled class distribution: {pd.Series(y_resampled).value_counts().to_dict()}")
        
        return pd.DataFrame(X_resampled, columns=X.columns), pd.Series(y_resampled)
    
    def train_models(self, X: pd.DataFrame, y: pd.Series) -> Dict[str, Any]:
        """Train multiple models with cross-validation"""
        
        print("Training models...")
        
        # Initialize models
        models = {
            'LightGBM': lgb.LGBMClassifier(**self.config.LIGHTGBM_PARAMS),
            'XGBoost': xgb.XGBClassifier(
                **self.config.XGBOOST_PARAMS,
                scale_pos_weight=(y == 0).sum() / (y == 1).sum()
            ),
            'CatBoost': CatBoostClassifier(**self.config.CATBOOST_PARAMS)
        }
        
        # Cross-validation setup
        skf = StratifiedKFold(
            n_splits=self.config.N_FOLDS, 
            shuffle=True, 
            random_state=self.config.RANDOM_STATE
        )
        
        # Train-validation split for final evaluation
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=self.config.TEST_SIZE, 
            random_state=self.config.RANDOM_STATE, stratify=y
        )
        
        results = {}
        
        for name, model in models.items():
            print(f"\nTraining {name}...")
            
            # Cross-validation
            cv_scores = cross_val_score(
                model, X_train, y_train, cv=skf, 
                scoring='roc_auc', n_jobs=-1
            )
            
            # Train on full training set
            if name == 'LightGBM':
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(50, verbose=False)]
                )
            elif name == 'XGBoost':
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    verbose=False
                )
            else:  # CatBoost
                model.fit(
                    X_train, y_train,
                    eval_set=(X_val, y_val),
                    verbose=False
                )
            
            # Predictions
            y_pred = model.predict(X_val)
            y_proba = model.predict_proba(X_val)[:, 1]
            
            # Optimize threshold
            best_threshold, best_f1 = Utils.optimize_threshold(y_val, y_proba)
            y_pred_optimized = (y_proba >= best_threshold).astype(int)
            
            # Metrics
            auc_score = roc_auc_score(y_val, y_proba)
            f1_optimized = f1_score(y_val, y_pred_optimized)
            
            results[name] = {
                'model': model,
                'cv_auc_mean': cv_scores.mean(),
                'cv_auc_std': cv_scores.std(),
                'val_auc': auc_score,
                'val_f1': f1_optimized,
                'threshold': best_threshold,
                'probabilities': y_proba
            }
            
            print(f"  CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
            print(f"  Val AUC: {auc_score:.4f}")
            print(f"  Val F1: {f1_optimized:.4f}")
            
            # Store feature importance
            if hasattr(model, 'feature_importances_'):
                self.feature_importance[name] = pd.DataFrame({
                    'feature': X.columns,
                    'importance': model.feature_importances_
                }).sort_values('importance', ascending=False)
        
        self.results = results
        return results
    
    def create_ensemble(self, X: pd.DataFrame, y: pd.Series) -> Dict[str, Any]:
        """Create ensemble models"""
        
        print("\nCreating ensemble models...")
        
        if not self.results:
            raise ValueError("No trained models found. Train models first.")
        
        # Weighted ensemble based on validation AUC
        weights = np.array([self.results[name]['val_auc'] for name in self.results.keys()])
        weights = weights / weights.sum()
        
        # Calculate weighted predictions
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=self.config.TEST_SIZE, 
            random_state=self.config.RANDOM_STATE, stratify=y
        )
        
        ensemble_proba = np.zeros(len(y_val))
        for i, (name, result) in enumerate(self.results.items()):
            ensemble_proba += weights[i] * result['probabilities']
        
        # Optimize ensemble threshold
        best_threshold, best_f1 = Utils.optimize_threshold(y_val, ensemble_proba)
        ensemble_pred = (ensemble_proba >= best_threshold).astype(int)
        ensemble_auc = roc_auc_score(y_val, ensemble_proba)
        
        # Voting ensemble
        base_estimators = [(name, result['model']) for name, result in self.results.items()]
        voting_clf = VotingClassifier(estimators=base_estimators, voting='soft', n_jobs=-1)
        
        # Cross-validation for voting classifier
        cv_scores = cross_val_score(
            voting_clf, X_train, y_train, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=self.config.RANDOM_STATE),
            scoring='roc_auc', n_jobs=-1
        )
        
        voting_clf.fit(X_train, y_train)
        voting_proba = voting_clf.predict_proba(X_val)[:, 1]
        voting_auc = roc_auc_score(y_val, voting_proba)
        
        ensemble_results = {
            'WeightedEnsemble': {
                'val_auc': ensemble_auc,
                'val_f1': best_f1,
                'threshold': best_threshold,
                'weights': weights,
                'probabilities': ensemble_proba
            },
            'VotingEnsemble': {
                'model': voting_clf,
                'cv_auc_mean': cv_scores.mean(),
                'cv_auc_std': cv_scores.std(),
                'val_auc': voting_auc,
                'probabilities': voting_proba
            }
        }
        
        print(f"Weighted Ensemble AUC: {ensemble_auc:.4f}")
        print(f"Voting Ensemble AUC: {voting_auc:.4f}")
        
        return ensemble_results

# ============================================================================
# MAIN PIPELINE CLASS
# ============================================================================
class ChurnPredictionPipeline:
    """Main pipeline orchestrator"""
    
    def __init__(self, config: Config):
        self.config = config
        self.feature_engineer = FeatureEngineer()
        self.model_trainer = ModelTrainer(config)
        self.best_model = None
        self.feature_columns = None
        self.scaler = None
        
        # Create output directory
        self.config.OUTPUT_DIR.mkdir(exist_ok=True)
    
    def run_training_pipeline(self):
        """Run the complete training pipeline"""
        
        print("=" * 80)
        print("TELECOM CHURN PREDICTION - OPTIMIZED PIPELINE")
        print("=" * 80)
        
        # 1. Load and explore data
        print("\n1. Loading training data...")
        df_train = pd.read_csv(self.config.TRAIN_PATH)
        print(f"Training data shape: {df_train.shape}")
        print(f"Class distribution: {df_train['label'].value_counts().to_dict()}")
        
        # 2. Feature engineering
        print("\n2. Feature engineering...")
        df_processed = self.feature_engineer.create_enhanced_features(df_train, is_train=True)
        
        # 3. Prepare features
        feature_cols = self._get_feature_columns(df_processed)
        self.feature_columns = Utils.clean_feature_names(feature_cols)
        
        X = df_processed[feature_cols].copy()
        X.columns = self.feature_columns
        y = df_processed['label']
        
        # 4. Handle missing values and infinite values
        X = self._handle_missing_values(X)
        
        # 5. Memory optimization
        X = Utils.reduce_memory_usage(X)
        
        # 6. Prepare data (feature selection + balancing)
        X_prepared, y_prepared = self.model_trainer.prepare_data(X, y)
        
        # 7. Train models
        model_results = self.model_trainer.train_models(X_prepared, y_prepared)
        
        # 8. Create ensembles
        ensemble_results = self.model_trainer.create_ensemble(X_prepared, y_prepared)
        
        # 9. Select best model
        all_results = {**model_results, **ensemble_results}
        best_model_name, best_model_info = self._select_best_model(all_results)
        
        print(f"\n🏆 Best Model: {best_model_name}")
        print(f"   Validation AUC: {best_model_info['val_auc']:.4f}")
        if 'val_f1' in best_model_info:
            print(f"   Validation F1: {best_model_info['val_f1']:.4f}")
        
        # 10. Save models and metadata
        self._save_training_artifacts(all_results, best_model_name, X.columns)
        
        return all_results, best_model_name
    
    def run_prediction_pipeline(self):
        """Run prediction on test data"""
        
        print("\n" + "=" * 80)
        print("PREDICTION PIPELINE")
        print("=" * 80)
        
        # Load test data
        print("Loading test data...")
        df_test = pd.read_csv(self.config.TEST_PATH)
        test_ids = df_test['id'].copy()
        
        # Feature engineering
        print("Creating features...")
        df_test_processed = self.feature_engineer.create_enhanced_features(df_test, is_train=False)
        
        # Prepare test features
        X_test = df_test_processed[self._get_feature_columns(df_test_processed)].copy()
        X_test.columns = self.feature_columns
        X_test = self._handle_missing_values(X_test)
        
        # Load best model
        best_model = self._load_best_model()
        
        # Make predictions
        print("Making predictions...")
        predictions = best_model.predict(X_test)
        probabilities = best_model.predict_proba(X_test)[:, 1]
        
        # Create submission
        submission = pd.DataFrame({
            'id': test_ids,
            'label': predictions
        })
        
        submission.to_csv(self.config.SUBMISSION_PATH, index=False)
        
        print(f"Predictions saved to {self.config.SUBMISSION_PATH}")
        print(f"Prediction distribution: {pd.Series(predictions).value_counts().to_dict()}")
        
        return submission
    
    def _get_feature_columns(self, df: pd.DataFrame) -> List[str]:
        """Get feature columns excluding metadata"""
        exclude_cols = ['Unnamed: 0', 'id', 'label', 'age_group']
        return [col for col in df.columns if col not in exclude_cols]
    
    def _handle_missing_values(self, X: pd.DataFrame) -> pd.DataFrame:
        """Handle missing values and infinite values"""
        print("Handling missing values...")
        
        for col in X.columns:
            if X[col].dtype in ['float64', 'int64', 'float32', 'int32']:
                X[col].fillna(X[col].median(), inplace=True)
            else:
                mode_val = X[col].mode()
                X[col].fillna(mode_val[0] if len(mode_val) > 0 else 0, inplace=True)
        
        # Replace infinite values
        X.replace([np.inf, -np.inf], 0, inplace=True)
        
        return X
    
    def _select_best_model(self, results: Dict[str, Any]) -> Tuple[str, Dict[str, Any]]:
        """Select the best performing model"""
        best_auc = 0
        best_name = ""
        best_info = {}
        
        for name, info in results.items():
            if info['val_auc'] > best_auc:
                best_auc = info['val_auc']
                best_name = name
                best_info = info
        
        return best_name, best_info
    
    def _save_training_artifacts(self, results: Dict[str, Any], best_model_name: str, 
                               feature_columns: List[str]):
        """Save models and metadata"""
        print("\nSaving training artifacts...")
        
        # Save best model
        best_model_info = results[best_model_name]
        if 'model' in best_model_info:
            with open(self.config.OUTPUT_DIR / f'best_model_{best_model_name.lower()}.pkl', 'wb') as f:
                pickle.dump(best_model_info['model'], f)
        
        # Save feature engineering artifacts
        artifacts = {
            'feature_columns': feature_columns,
            'percentile_maps': self.feature_engineer.percentile_maps,
            'best_model_name': best_model_name,
            'results_summary': results
        }
        
        with open(self.config.OUTPUT_DIR / 'training_artifacts.pkl', 'wb') as f:
            pickle.dump(artifacts, f)
        
        print("✅ Training artifacts saved successfully")
    
    def _load_best_model(self):
        """Load the best trained model"""
        with open(self.config.OUTPUT_DIR / 'training_artifacts.pkl', 'rb') as f:
            artifacts = pickle.load(f)
        
        best_model_name = artifacts['best_model_name']
        
        with open(self.config.OUTPUT_DIR / f'best_model_{best_model_name.lower()}.pkl', 'rb') as f:
            model = pickle.load(f)
        
        return model

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    # Initialize configuration
    config = Config()
    
    # Initialize pipeline
    pipeline = ChurnPredictionPipeline(config)
    
    results, best_model_name = pipeline.run_training_pipeline()
    
    # Run prediction pipeline
    submission = pipeline.run_prediction_pipeline()
    
    print("\n" + "=" * 80)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    print(f"🎯 Best Model: {best_model_name}")
    print(f"📄 Submission saved to: {config.SUBMISSION_PATH}")
    print("=" * 80)
        

TELECOM CHURN PREDICTION - OPTIMIZED PIPELINE

1. Loading training data...
Training data shape: (59904, 88)
Class distribution: {0: 44904, 1: 15000}

2. Feature engineering...
Starting feature engineering... Initial shape: (59904, 88)
Feature engineering completed. Final shape: (59904, 129)
Handling missing values...
Memory usage decreased from 57.59 MB to 28.96 MB (49.7% reduction)
Preparing data...
Starting feature selection from 126 features...
Selected 64 features
Original class distribution: {0: 44904, 1: 15000}
Resampled class distribution: {0: 38656, 1: 38656}
Training models...

Training LightGBM...
  CV AUC: 0.9515 ± 0.0015
  Val AUC: 0.9504
  Val F1: 0.8741

Training XGBoost...
  CV AUC: 0.9526 ± 0.0016
  Val AUC: 0.9523
  Val F1: 0.8759

Training CatBoost...
  CV AUC: 0.9498 ± 0.0012
  Val AUC: 0.9492
  Val F1: 0.8739

Creating ensemble models...
Weighted Ensemble AUC: 0.9518
Voting Ensemble AUC: 0.9518

🏆 Best Model: XGBoost
   Validation AUC: 0.9523
   Validation F1: 0.875

ValueError: feature_names mismatch: ['sex', 'age', 'innet_dura', 'arpu', 'l3m_avg_mou', 'l3m_avg_dou', 'cm_local_voice_dura', 'cm_flux_use', 'flux_4g_use', 'flux_up_4g_sum', 'flux_down_4g_sum', 'wday_day_flux', 'wday_night_flux', 'nwday_day_flux', 'nwday_night_flux', 'cm_flux_tot_cnt', 'cm_base_plan_flux', 'cm_base_plan_flux_use', 'gprs_days', 'is_10g_pon', 'bd_flux_m', 'bd_dur_m', 'bd_cnt_m', 'is_bd_tv', 'user_duration_m', 'login_times_m', 'click_times_m', 'watch_times_m', 'open_day_m', 'click_day_m', 'term_cont_mon', 'grp_cnts', 'if_nulim_prod', 'out_call', 'video_cnt_m', 'music_cnt_m', 'gm_usr_lbl', 'arpu_per_mb', 'usage_efficiency', 'tenure_months', 'total_flux', 'flux_variance', 'flux_skewness', 'video_total', 'video_cnt_m_ratio', 'voice_data_ratio', 'engagement_score', 'arpu_log1p', 'arpu_sqrt', 'arpu_percentile', 'cm_flux_use_log1p', 'cm_flux_use_sqrt', 'cm_flux_use_percentile', 'innet_dura_log1p', 'innet_dura_sqrt', 'innet_dura_percentile', 'age_log1p', 'age_sqrt', 'age_percent